# Text Vectorization: TF-IDF

**References followed:**
1. [Text Vectorization: Term Frequency – Inverse Document Frequency (TF-IDF)](https://towardsdatascience.com/text-vectorization-term-frequency-inverse-document-frequency-tfidf-5a3f9604da6d)
2. [Text Vectorization Using Python: TF-IDF (Okan Bulut)](https://okan.cloud/posts/2022-01-16-text-vectorization-using-python-tf-idf/)

**Concepts covered:**
- **TF (Term Frequency)** = (count of word in document) / (total words in document)
- **IDF (Inverse Document Frequency)** = log( total documents / documents containing the word )
- **TF-IDF** = TF × IDF

This notebook reproduces the pipeline from reference (2): load text data → preprocess (clean, lowercase, remove stopwords, lemmatize) → vectorize with `TfidfVectorizer` → inspect the resulting matrix → sanity-check the math manually.

*Note: The original blog uses the Kaggle "Short Answer Scoring" dataset (Essay Set 3 — pandas vs. koalas vs. pythons). That exact file is hosted on GitHub; if you have internet access to `raw.githubusercontent.com`, you can swap it in (see the commented cell below). This notebook uses a small hand-written dataset with the same theme and structure so the full pipeline still runs anywhere.*

In [1]:
import re
import math
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

## Step 1: Load the data

(Optional) To use the *real* dataset from the blog instead, uncomment the cell below.

In [2]:
# --- OPTIONAL: use the real dataset from the blog (requires internet access) ---
# import pandas as pd
# with open('train_rel_2.tsv', 'r') as f:
#     lines = f.readlines()
#     response, score = [], []
#     for line in lines[1:]:
#         temp = line.split('\t')
#         if temp[1] == '3':
#             response.append(temp[-1])
#             score.append(int(temp[2]))
# data = pd.DataFrame(list(zip(response, score)), columns=['response', 'score'])
# print(data.shape)

In [3]:
# Sample dataset (same theme as the blog's Essay Set 3: pandas vs koalas vs pythons)
responses = [
    "China's panda and Australia's koala are two animals that aren't predators, "
    "pandas eat bamboo and koala's eat eucalyptus leaves. Therefore, they are "
    "harmless. They are both different from pythons because pythons are "
    "potentially dangerous considering they can swallow an entire alligator.",

    "Pandas and koalas are similar because they are both specialist eaters, "
    "unlike pythons which are generalists and eat many different animals "
    "including alligators.",

    "Pandas in China and Koalas in Australia are similar because they only "
    "eat one type of food, panda's eat bamboo and koala's eat eucalyptus, "
    "while pythons will eat almost anything.",

    "Pandas in China only eat bamboo and Koalas in Australia only eat "
    "eucalyptus leaves so they are both specialists, but pythons are "
    "generalists since they eat many kinds of animals.",

    "Pandas in China and koalas from Australia are similar to each other "
    "and are different from pythons in China and Africa because pythons eat "
    "many different animals like alligators, while pandas and koalas each "
    "eat only one type of food.",
]
scores = [1, 1, 1, 2, 0]

data = pd.DataFrame(list(zip(responses, scores)), columns=['response', 'score'])
print("Shape:", data.shape)
data.head()

Shape: (5, 2)


,response,score
0,China's panda and Australia's koala are two an...,1
1,Pandas and koalas are similar because they are...,1
2,Pandas in China and Koalas in Australia are si...,1
3,Pandas in China only eat bamboo and Koalas in ...,2
4,Pandas in China and koalas from Australia are ...,0


## Step 2: Preprocessing

Before TF-IDF, we clean the text:
- remove non-alphabetic characters (punctuation, numbers)
- lowercase everything
- tokenize (split into words)
- remove **stopwords** (common words like "the", "is", "and" that carry little meaning)
- **lemmatize** each word (reduce to its dictionary/base form, e.g. "eating" → "eat")

This matters because raw stopword-heavy text makes documents look artificially similar, and it also inflates the number of dimensions (the "curse of dimensionality").

In [4]:
stopwords = nltk.corpus.stopwords.words('english')
lemmatizer = WordNetLemmatizer()

data_without_stopwords = []
for i in range(len(responses)):
    doc = re.sub('[^a-zA-Z]', ' ', responses[i])
    doc = doc.lower()
    doc = doc.split()
    doc = [lemmatizer.lemmatize(word) for word in doc if word not in set(stopwords)]
    doc = ' '.join(doc)
    data_without_stopwords.append(doc)

print("BEFORE:", data.response[0])
print()
print("AFTER: ", data_without_stopwords[0])

BEFORE: China's panda and Australia's koala are two animals that aren't predators, pandas eat bamboo and koala's eat eucalyptus leaves. Therefore, they are harmless. They are both different from pythons because pythons are potentially dangerous considering they can swallow an entire alligator.

AFTER:  china panda australia koala two animal predator panda eat bamboo koala eat eucalyptus leaf therefore harmless different python python potentially dangerous considering swallow entire alligator


## Step 3: Vectorize with TF-IDF

`sklearn.feature_extraction.text.TfidfVectorizer` builds the vocabulary and computes TF-IDF weights for every word in every document in one call.

In [5]:
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(data_without_stopwords)

print("n_samples: %d, n_features: %d" % vectors.shape)

n_samples: 5, n_features: 37


## Step 4: Inspect the TF-IDF matrix

Let's look at the top words (by total TF-IDF weight summed across all documents) and how their weight varies per document.

In [6]:
tf_idf = pd.DataFrame(vectors.todense())
tf_idf.columns = vectorizer.get_feature_names_out()
tfidf_matrix = tf_idf.T
tfidf_matrix.columns = ['response' + str(i) for i in range(1, len(responses) + 1)]
tfidf_matrix['count'] = tfidf_matrix.sum(axis=1)

top_words = tfidf_matrix.sort_values(by='count', ascending=False).drop(columns=['count']).head(10)
top_words

,response1,response2,response3,response4,response5
eat,0.238029,0.176103,0.559398,0.477164,0.268165
koala,0.238029,0.176103,0.279699,0.159055,0.268165
panda,0.238029,0.176103,0.279699,0.159055,0.268165
python,0.238029,0.176103,0.139849,0.159055,0.268165
china,0.140713,0.000000,0.165347,0.188054,0.317057
different,0.167271,0.247507,0.000000,0.000000,0.376896
animal,0.140713,0.208211,0.000000,0.188054,0.158529
many,0.000000,0.247507,0.000000,0.223546,0.188448
australia,0.140713,0.000000,0.165347,0.188054,0.158529
similar,0.000000,0.247507,0.196553,0.000000,0.188448


**Reading this table:** each cell is the TF-IDF weight of that word in that document. A 0 means the word doesn't appear in that document at all. Words that show up in *every* document (like "eat", "panda", "koala" here, since all responses are on the same topic) still get *some* weight because TF-IDF is per-document — but as we'll see below, if a word appeared in literally every document with the same relative frequency, its IDF component pulls its importance down.

## Step 5: Manual sanity check (understanding TF and IDF separately)

To make sure we actually understand the formulas — not just call a library — let's compute TF and IDF by hand for two words and compare.

- **TF** = (word count in doc) / (total words in doc)
- **IDF** = log( total docs / docs containing the word )
- **TF-IDF** = TF × IDF

In [7]:
def manual_tf(word, doc_tokens):
    return doc_tokens.count(word) / len(doc_tokens)

def manual_idf(word, all_docs_tokens):
    n_docs_with_word = sum(1 for doc in all_docs_tokens if word in doc)
    return math.log(len(all_docs_tokens) / n_docs_with_word)

all_docs_tokens = [doc.split() for doc in data_without_stopwords]

for sample_word in ["panda", "bamboo"]:
    doc_idx = 0
    tf_val = manual_tf(sample_word, all_docs_tokens[doc_idx])
    idf_val = manual_idf(sample_word, all_docs_tokens)
    n_docs_with_word = sum(1 for doc in all_docs_tokens if sample_word in doc)

    print(f"Word: '{sample_word}'")
    print(f"  Appears in {n_docs_with_word} of {len(all_docs_tokens)} documents")
    print(f"  TF (in response 1)  = {tf_val:.4f}")
    print(f"  IDF                 = {idf_val:.4f}")
    print(f"  TF x IDF (manual)   = {tf_val * idf_val:.4f}")
    print()

Word: 'panda'
  Appears in 5 of 5 documents
  TF (in response 1)  = 0.0800
  IDF                 = 0.0000
  TF x IDF (manual)   = 0.0000

Word: 'bamboo'
  Appears in 3 of 5 documents
  TF (in response 1)  = 0.0400
  IDF                 = 0.5108
  TF x IDF (manual)   = 0.0204



**Takeaway:** "panda" appears in *all* documents here, so its IDF = log(5/5) = **0** — meaning it contributes nothing to distinguishing documents from each other, even though it's clearly on-topic. "bamboo" appears in only 3 of 5 documents, so it keeps a nonzero IDF and therefore a nonzero TF-IDF score.

This is exactly the core idea from the blog: **TF-IDF down-weights words that are common across the whole corpus, and up-weights words that are distinctive to a smaller set of documents** — giving a more meaningful representation than simple word counts (the term-document / bag-of-words matrix).

Note: `sklearn`'s `TfidfVectorizer` uses a *smoothed* IDF formula by default (adds 1 to numerator and denominator to avoid divide-by-zero) and L2-normalizes each document's vector — so its exact numbers won't match this manual, textbook calculation, but the underlying logic and relative ranking of word importance is the same.